In [10]:
import pandas as pd
import numpy as np
df=pd.read_csv("gpu_forecasting_dataset.csv")
df.sample(20)

,snapshot_date,provider,gpu_model,avg_price,day_of_week,month,day,price_lag_1,price_lag_2,price_lag_3
291,2026-07-14,AtmosCompute,A40,1.100000,1,7,14,1.100000,1.100000,1.100000
1193,2026-07-10,Crusoe,MI300X,3.450000,4,7,10,NaN,NaN,NaN
1947,2026-07-13,Sesterce,RTX PRO 6000,1.303500,0,7,13,1.303500,1.303500,1.303500
1501,2026-07-13,Salad Cloud,RTX 4090,0.160000,0,7,13,0.160000,0.160000,0.160000
1962,2026-07-13,Amazon AWS,TESLA T4,0.827333,0,7,13,0.827333,0.827333,0.827333
654,2026-07-10,Microsoft Azure,H100 SXM,5.155260,4,7,10,NaN,NaN,NaN
1436,2026-07-12,RunPod,RTX 4080,0.270000,6,7,12,0.270000,0.270000,NaN
718,2026-07-13,Verda,H100 SXM,2.193813,0,7,13,2.193812,2.193813,2.193813
1000,2026-07-11,Sesterce,L4,1.045000,5,7,11,1.045000,NaN,NaN
958,2026-07-14,AceCloud,L4,0.730533,1,7,14,0.730533,0.730533,0.730533


In [5]:
print(df.describe())
print(df.groupby("gpu_model")["avg_price"].mean())

         avg_price  day_of_week   month          day  price_lag_1  price_lag_2
count  1222.000000  1222.000000  1222.0  1222.000000   801.000000   395.000000
mean      2.114472     4.995090     7.0    10.995090     2.141710     2.160060
std       2.465106     0.817484     0.0     0.817484     2.478273     2.488321
min       0.040000     4.000000     7.0    10.000000     0.040000     0.040000
25%       0.516785     4.000000     7.0    10.000000     0.535000     0.539500
50%       1.300000     5.000000     7.0    11.000000     1.340500     1.362500
75%       2.700000     6.000000     7.0    12.000000     2.737875     2.747450
max      18.000000     6.000000     7.0    12.000000    18.000000    18.000000
gpu_model
A10                   1.343138
A100 PCIE             1.347105
A100 SXM              1.801920
A16                   0.549820
A2                    0.282600
A30                   0.671603
A40                   1.040537
B200                  5.792876
GAUDI 2               1.230000


In [6]:
print(df.columns.tolist())

['snapshot_date', 'provider', 'gpu_model', 'avg_price', 'day_of_week', 'month', 'day', 'price_lag_1', 'price_lag_2']


In [8]:
len(df)

2034

In [11]:
lag_cols = ['price_lag_1', 'price_lag_2', 'price_lag_3']
lags = df[lag_cols].dropna()

geom_mean = np.exp(np.log(lags).mean(axis=1))

geo_diff = np.sqrt(
    (lags['price_lag_1'] - lags['price_lag_2'])**2 +
    (lags['price_lag_2'] - lags['price_lag_3'])**2 +
    (lags['price_lag_3'] - lags['price_lag_1'])**2
) / np.sqrt(3)

result = lags.assign(
    geom_mean=geom_mean,
    geo_diff=geo_diff,
    diff_12=lags['price_lag_1'] - lags['price_lag_2'],
    diff_23=lags['price_lag_2'] - lags['price_lag_3'],
    diff_31=lags['price_lag_3'] - lags['price_lag_1']
)

print(result[['price_lag_1', 'price_lag_2', 'price_lag_3', 'geom_mean', 'geo_diff']].head())
print("\nGeometric difference summary:")
print(result['geo_diff'].describe())

    price_lag_1  price_lag_2  price_lag_3  geom_mean  geo_diff
3      1.486667     1.486667     1.486667   1.486667       0.0
4      1.486667     1.486667     1.486667   1.486667       0.0
8      1.051200     1.051200     1.051200   1.051200       0.0
9      1.051200     1.051200     1.051200   1.051200       0.0
13     1.290000     1.290000     1.290000   1.290000       0.0

Geometric difference summary:
count    7.940000e+02
mean     8.831898e-03
std      6.321741e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      9.064933e-17
max      8.899813e-01
Name: geo_diff, dtype: float64


In [ ]:
# check how many rows have no change across the three lags (using result which has only rows with all 3 lags)
is_constant = np.isclose(result['price_lag_1'], result['price_lag_2']) & np.isclose(result['price_lag_2'], result['price_lag_3'])
total = len(result)
num_constant = is_constant.sum()
print(f"{num_constant} / {total} rows ({num_constant/total:.1%}) have no change across the three lags")

# show per gpu_model (and provider) which ones never change or mostly unchanged
subset = df.loc[result.index].assign(constant=is_constant.values)
group = subset.groupby(['gpu_model','provider'])['constant'].agg(['sum','count'])
group['pct_constant'] = group['sum'] / group['count']
print(group.sort_values('pct_constant', ascending=False).head(30))